
# Advanced LLM Training Infrastructure for Interview Prep
## ZeRO, Offload, Tensor / Sequence / Expert Parallelism, Multi-Node Comms, Distributed Checkpointing, Observability, and Failure Triage

This notebook is the **advanced follow-on** to a fundamentals training-infrastructure notebook.

It is designed for:
- LLM training infrastructure interviews
- senior ML systems interviews
- L5 / L6-style distributed training discussions
- candidates who want **concepts + intuition + production-inspired code**

## What this notebook covers

By the end, you will have:
1. ZeRO stage and offload memory-intuition tooling
2. exact toy demos for tensor parallel and sequence parallel ideas
3. a simple MoE / expert-parallel routing simulation
4. communication-cost intuition across fabrics
5. NCCL debugging helpers and synthetic log triage
6. a local distributed-checkpoint analogy with shard files + manifest
7. step-time observability and a tiny profiler pass
8. failure-scenario triage exercises with suggested mitigations

## Important scope note

This notebook is built for **single-process Colab-style environments**.

That means:
- the code is runnable locally
- the ideas are production-inspired
- but real multi-node launches are discussed conceptually rather than fully executed

That is intentional. The goal is interview-grade understanding.



## Why this notebook exists

A fundamentals notebook can teach:
- DDP
- FSDP
- AMP
- activation checkpointing
- pipeline-parallel basics

But advanced training-infra interviews often go further:
- *When would you choose ZeRO-2 vs ZeRO-3?*
- *What exactly does offload buy you and what does it cost?*
- *Why do tensor parallel and sequence parallel often appear together?*
- *What is expert parallelism solving in MoE training?*
- *How do you think about NCCL issues before touching code?*
- *Why do distributed checkpoints matter for changing world sizes?*
- *What metrics would you watch before opening a heavy profiler?*

This notebook is about those questions.


In [ ]:

# ============================================================
# Imports and environment setup
# ============================================================
# This notebook is an advanced training-infrastructure notebook.
# It is designed to run on Colab or CPU-only systems, so many of the
# demos are local or simulation-based. That is intentional.
#
# The goal is to teach the *ideas* behind:
# - ZeRO and offload
# - tensor / sequence / expert parallelism
# - multi-node communication intuition
# - NCCL debugging
# - distributed checkpointing
# - observability and step-time analysis
# - real failure triage patterns

import copy
import json
import math
import os
import random
import shutil
import statistics
import tempfile
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available. CPU path is still fine for this notebook.")



## 1. Shared toy model

We keep one modest decoder-style model around so multiple sections have something concrete to refer to:
- memory accounting
- checkpoint sharding
- observability and profiling
- failure triage examples


In [ ]:

# ============================================================
# Small decoder-style LM used in several sections
# ============================================================
# We keep one modest-sized model so:
# - memory accounting has something concrete to refer to
# - distributed checkpointing can shard a real state_dict
# - observability and profiler sections can touch realistic ops
#
# The model is intentionally smaller than a real LLM because
# this notebook is for teaching and Colab-friendly execution.

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
DIGIT_OFFSET = 3
VOCAB_SIZE = 13

def make_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    """Return a lower-triangular causal mask."""
    return torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device))

def make_counting_batch(
    batch_size: int,
    min_digits: int = 6,
    max_digits: int = 12,
    device: torch.device = torch.device("cpu"),
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Build a tiny next-token language-model batch."""
    rows = []
    max_len = 0

    for _ in range(batch_size):
        start_digit = random.randint(0, 9)
        seq_len = random.randint(min_digits, max_digits)

        digits = [DIGIT_OFFSET + ((start_digit + i) % 10) for i in range(seq_len)]
        full_sequence = [BOS_ID] + digits + [EOS_ID]

        rows.append(full_sequence)
        max_len = max(max_len, len(full_sequence))

    padded_rows = [
        row + [PAD_ID] * (max_len - len(row))
        for row in rows
    ]

    full = torch.tensor(padded_rows, dtype=torch.long, device=device)
    input_ids = full[:, :-1]
    target_ids = full[:, 1:]
    return input_ids, target_ids

class TinyCausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        return x.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, _, seq_len, _ = x.shape
        return x.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        q = self._split_heads(self.q_proj(x))
        k = self._split_heads(self.k_proj(x))
        v = self._split_heads(self.v_proj(x))

        seq_len = x.size(1)
        attn_mask = make_causal_mask(seq_len=seq_len, device=x.device)

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=False,
        )

        y = self._merge_heads(y)
        y = self.resid_dropout(self.out_proj(y))
        return y

class TinyFeedForward(nn.Module):
    def __init__(self, d_model: int, ff_dim: int, dropout: float = 0.0):
        super().__init__()
        self.fc1 = nn.Linear(d_model, ff_dim)
        self.fc2 = nn.Linear(ff_dim, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class TinyDecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, ff_dim: int, dropout: float = 0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = TinyCausalSelfAttention(d_model=d_model, num_heads=num_heads, dropout=dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = TinyFeedForward(d_model=d_model, ff_dim=ff_dim, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.dropout(self.attn(self.ln1(x)))
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

class TinyCausalLM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 48,
        num_heads: int = 4,
        ff_dim: int = 96,
        num_layers: int = 3,
        dropout: float = 0.0,
        max_len: int = 96,
    ):
        super().__init__()

        self.d_model = d_model
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.embed_dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            TinyDecoderBlock(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout)
            for _ in range(num_layers)
        ])

        self.final_ln = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embed.weight

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = input_ids.shape

        positions = torch.arange(seq_len, device=input_ids.device)
        x = self.token_embed(input_ids) * math.sqrt(self.d_model)
        x = x + self.pos_embed(positions)[None, :, :]
        x = self.embed_dropout(x)

        for layer in self.layers:
            x = layer(x)

        x = self.final_ln(x)
        logits = self.lm_head(x)
        return logits

toy_model = TinyCausalLM(
    vocab_size=VOCAB_SIZE,
    d_model=48,
    num_heads=4,
    ff_dim=96,
    num_layers=3,
    dropout=0.0,
    max_len=96,
).to(device)

print("Toy model parameter count:", sum(p.numel() for p in toy_model.parameters()))



## 2. ZeRO and offload intuition

### Mental model

At large scale, memory pressure comes from more than just model parameters:
- parameters
- gradients
- optimizer state
- master parameters
- activations
- fragmentation / temporary buffers

ZeRO attacks the *replication* side of that problem by sharding model state across data-parallel ranks.

### Stage summary
- **Stage 0**: replicate everything
- **Stage 1**: shard optimizer state
- **Stage 2**: shard optimizer state + gradients
- **Stage 3**: shard optimizer state + gradients + parameters

### Offload summary
Offload moves some state to CPU or slower storage:
- less accelerator memory
- more communication / latency pressure
- often useful when memory is the main blocker


In [ ]:

# ============================================================
# ZeRO / optimizer sharding / offload intuition
# ============================================================
# The point of this section is not to run DeepSpeed.
# The point is to develop a concrete memory-accounting mental model.

@dataclass
class MemoryAssumptions:
    param_bytes: int = 2
    grad_bytes: int = 4
    master_param_bytes: int = 4
    adam_m_bytes: int = 4
    adam_v_bytes: int = 4

@dataclass
class ZeROMemoryEstimate:
    stage: int
    world_size: int
    gpu_param_mb: float
    gpu_grad_mb: float
    gpu_optim_mb: float
    cpu_or_nvme_mb: float
    total_gpu_mb: float

def count_trainable_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def bytes_to_mb(num_bytes: float) -> float:
    return num_bytes / (1024 ** 2)

def estimate_zero_memory(
    num_params: int,
    world_size: int,
    stage: int,
    assumptions: MemoryAssumptions,
    offload_optimizer: bool = False,
    offload_params: bool = False,
) -> ZeROMemoryEstimate:
    full_param = num_params * assumptions.param_bytes
    full_grad = num_params * assumptions.grad_bytes
    full_optim = num_params * (
        assumptions.master_param_bytes
        + assumptions.adam_m_bytes
        + assumptions.adam_v_bytes
    )

    gpu_param = full_param
    gpu_grad = full_grad
    gpu_optim = full_optim
    cpu_or_nvme = 0.0

    if stage >= 1:
        gpu_optim = full_optim / world_size
    if stage >= 2:
        gpu_grad = full_grad / world_size
    if stage >= 3:
        gpu_param = full_param / world_size

    if offload_optimizer:
        cpu_or_nvme += gpu_optim
        gpu_optim = 0.0
    if offload_params:
        cpu_or_nvme += gpu_param
        gpu_param = 0.0

    total_gpu = gpu_param + gpu_grad + gpu_optim

    return ZeROMemoryEstimate(
        stage=stage,
        world_size=world_size,
        gpu_param_mb=bytes_to_mb(gpu_param),
        gpu_grad_mb=bytes_to_mb(gpu_grad),
        gpu_optim_mb=bytes_to_mb(gpu_optim),
        cpu_or_nvme_mb=bytes_to_mb(cpu_or_nvme),
        total_gpu_mb=bytes_to_mb(total_gpu),
    )

num_params = count_trainable_parameters(toy_model)
assumptions = MemoryAssumptions()

zero_rows = [
    estimate_zero_memory(num_params, 8, stage, assumptions, False, False)
    for stage in [0, 1, 2, 3]
]

print("Rough per-rank GPU memory by ZeRO stage")
for row in zero_rows:
    print(asdict(row))

plt.figure(figsize=(7, 4))
plt.bar([f"stage {row.stage}" for row in zero_rows], [row.total_gpu_mb for row in zero_rows])
plt.ylabel("Per-rank GPU memory (MB)")
plt.title("ZeRO stage intuition")
plt.tight_layout()
plt.show()

offload_variants = [
    estimate_zero_memory(num_params, 8, 3, assumptions, False, False),
    estimate_zero_memory(num_params, 8, 3, assumptions, True, False),
    estimate_zero_memory(num_params, 8, 3, assumptions, True, True),
]

plt.figure(figsize=(8, 4))
plt.bar(
    ["stage3_no_offload", "stage3_optim_offload", "stage3_param+optim_offload"],
    [row.total_gpu_mb for row in offload_variants],
)
plt.ylabel("Per-rank GPU memory (MB)")
plt.title("Offload intuition")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

deepspeed_zero2_offload_config = {
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {"device": "cpu"},
    }
}

deepspeed_zero3_offload_config = {
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu"},
        "offload_param": {"device": "cpu"},
    }
}

print("Example ZeRO-2 optimizer offload config")
print(json.dumps(deepspeed_zero2_offload_config, indent=2))

print("\nExample ZeRO-3 parameter+optimizer offload config")
print(json.dumps(deepspeed_zero3_offload_config, indent=2))



## 3. Tensor parallelism, sequence parallelism, and expert parallelism

### Tensor parallelism
Split the math of one layer across devices.

### Sequence parallelism
Split token-local work across sequence positions.

### Expert parallelism
In Mixture-of-Experts training, route tokens to selected experts and worry about load balance and capacity.


In [ ]:

# ============================================================
# Tensor parallelism, sequence parallelism, expert parallelism
# ============================================================

class ColumnParallelLinearDemo(nn.Module):
    def __init__(self, full_linear: nn.Linear, world_size: int):
        super().__init__()
        self.in_features = full_linear.in_features
        self.out_features = full_linear.out_features
        self.world_size = world_size

        assert self.out_features % world_size == 0
        shard_out = self.out_features // world_size

        self.weight_shards = nn.ParameterList()
        self.bias_shards = nn.ParameterList()

        for rank in range(world_size):
            start = rank * shard_out
            end = (rank + 1) * shard_out
            self.weight_shards.append(nn.Parameter(full_linear.weight.data[start:end].clone()))
            self.bias_shards.append(nn.Parameter(full_linear.bias.data[start:end].clone()))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        parts = []
        for w, b in zip(self.weight_shards, self.bias_shards):
            parts.append(F.linear(x, w, b))
        return torch.cat(parts, dim=-1)

class RowParallelLinearDemo(nn.Module):
    def __init__(self, full_linear: nn.Linear, world_size: int):
        super().__init__()
        self.in_features = full_linear.in_features
        self.out_features = full_linear.out_features
        self.world_size = world_size

        assert self.in_features % world_size == 0
        shard_in = self.in_features // world_size

        self.weight_shards = nn.ParameterList()
        self.bias = nn.Parameter(full_linear.bias.data.clone())

        for rank in range(world_size):
            start = rank * shard_in
            end = (rank + 1) * shard_in
            self.weight_shards.append(nn.Parameter(full_linear.weight.data[:, start:end].clone()))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_shards = torch.chunk(x, chunks=self.world_size, dim=-1)
        partials = []
        for x_shard, w in zip(x_shards, self.weight_shards):
            partials.append(F.linear(x_shard, w, bias=None))
        return sum(partials) + self.bias

torch.manual_seed(SEED)
full_linear_col = nn.Linear(12, 8)
col_demo = ColumnParallelLinearDemo(full_linear_col, world_size=2)
x_col = torch.randn(4, 12)
full_y_col = full_linear_col(x_col)
sharded_y_col = col_demo(x_col)
col_equal = torch.allclose(full_y_col, sharded_y_col, atol=1e-6, rtol=1e-5)

torch.manual_seed(SEED)
full_linear_row = nn.Linear(12, 8)
row_demo = RowParallelLinearDemo(full_linear_row, world_size=2)
x_row = torch.randn(4, 12)
full_y_row = full_linear_row(x_row)
sharded_y_row = row_demo(x_row)
row_equal = torch.allclose(full_y_row, sharded_y_row, atol=1e-6, rtol=1e-5)

print("Column-parallel equivalence:", col_equal)
print("Row-parallel equivalence:", row_equal)

# Sequence parallel intuition:
# Token-wise LayerNorm is an easy exact example because each token
# can be normalized independently over the hidden dimension.
seq_tensor = torch.randn(2, 8, 16)
layer_norm = nn.LayerNorm(16)

full_seq_ln = layer_norm(seq_tensor)
seq_shards = torch.chunk(seq_tensor, chunks=2, dim=1)
seq_outputs = [layer_norm(shard) for shard in seq_shards]
seq_parallel_out = torch.cat(seq_outputs, dim=1)

seq_equal = torch.allclose(full_seq_ln, seq_parallel_out, atol=1e-6, rtol=1e-5)
print("Sequence-parallel token-wise LayerNorm equivalence:", seq_equal)

plt.figure(figsize=(7, 3))
plt.plot(full_seq_ln[0, 0].detach().numpy(), label="full")
plt.plot(seq_parallel_out[0, 0].detach().numpy(), linestyle="--", label="sequence_parallel")
plt.title("Sequence-parallel intuition on LayerNorm")
plt.legend()
plt.tight_layout()
plt.show()

@dataclass
class MoERoutingResult:
    assignments: List[int]
    counts_per_expert: Dict[int, int]
    dropped_tokens: int
    load_balance_score: float

class SimpleMoERouter(nn.Module):
    def __init__(self, hidden_dim: int, num_experts: int, capacity_factor: float = 1.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_experts = num_experts
        self.capacity_factor = capacity_factor
        self.router = nn.Linear(hidden_dim, num_experts)

    def forward(self, x: torch.Tensor) -> MoERoutingResult:
        logits = self.router(x)
        probs = torch.softmax(logits, dim=-1)
        assignments = probs.argmax(dim=-1).tolist()

        num_tokens = x.size(0)
        capacity = math.ceil((num_tokens / self.num_experts) * self.capacity_factor)

        counts = {expert_idx: 0 for expert_idx in range(self.num_experts)}
        dropped = 0
        accepted_assignments = []

        for expert_idx in assignments:
            if counts[expert_idx] < capacity:
                counts[expert_idx] += 1
                accepted_assignments.append(expert_idx)
            else:
                dropped += 1
                accepted_assignments.append(-1)

        count_values = list(counts.values())
        mean_count = statistics.mean(count_values)
        stdev_count = statistics.pstdev(count_values) if len(count_values) > 1 else 0.0
        load_balance_score = stdev_count / mean_count if mean_count > 0 else 0.0

        return MoERoutingResult(
            assignments=accepted_assignments,
            counts_per_expert=counts,
            dropped_tokens=dropped,
            load_balance_score=load_balance_score,
        )

torch.manual_seed(SEED)
moe_router = SimpleMoERouter(hidden_dim=16, num_experts=4, capacity_factor=1.0)
moe_input = torch.randn(24, 16)
moe_result = moe_router(moe_input)

print("MoE counts per expert:", moe_result.counts_per_expert)
print("Dropped tokens:", moe_result.dropped_tokens)
print("Load-balance score:", round(moe_result.load_balance_score, 4))

plt.figure(figsize=(6, 4))
plt.bar(
    [f"expert_{idx}" for idx in moe_result.counts_per_expert.keys()],
    list(moe_result.counts_per_expert.values()),
)
plt.ylabel("Accepted tokens")
plt.title("Expert-parallel routing load")
plt.tight_layout()
plt.show()



## 4. Multi-node communication and NCCL debugging intuition

A lot of distributed-training pain happens outside the Python model code:
- communication fabric limits
- startup latency
- rank mismatches
- one crashed rank blocking everyone else
- interface selection problems

That is why infra engineers need both:
- communication-cost intuition
- debugging discipline


In [ ]:

# ============================================================
# Multi-node communication and NCCL debugging intuition
# ============================================================

@dataclass
class CommFabric:
    name: str
    bandwidth_GBps: float
    latency_us: float

def estimate_allreduce_time_ms(message_mb: float, world_size: int, fabric: CommFabric) -> float:
    p = world_size
    alpha_ms = fabric.latency_us / 1000.0
    transfer_ms = message_mb / fabric.bandwidth_GBps
    factor = 2.0 * (p - 1) / p
    return factor * (alpha_ms + transfer_ms)

def estimate_allgather_time_ms(message_mb: float, world_size: int, fabric: CommFabric) -> float:
    p = world_size
    alpha_ms = fabric.latency_us / 1000.0
    transfer_ms = message_mb / fabric.bandwidth_GBps
    factor = (p - 1) / p
    return factor * (alpha_ms + transfer_ms)

fabrics = [
    CommFabric(name="single_node_fast_fabric", bandwidth_GBps=300.0, latency_us=2.0),
    CommFabric(name="multi_node_rdma", bandwidth_GBps=50.0, latency_us=8.0),
    CommFabric(name="multi_node_tcp", bandwidth_GBps=12.0, latency_us=30.0),
]

message_sizes_mb = [1, 4, 16, 64, 256]
world_size = 8

for fabric in fabrics:
    print(f"Fabric: {fabric.name}")
    for msg_mb in message_sizes_mb:
        ar = estimate_allreduce_time_ms(msg_mb, world_size, fabric)
        ag = estimate_allgather_time_ms(msg_mb, world_size, fabric)
        print(f"  msg={msg_mb:>3} MB  all_reduce={ar:7.3f} ms  all_gather={ag:7.3f} ms")
    print()

plt.figure(figsize=(8, 4))
for fabric in fabrics:
    ys = [estimate_allreduce_time_ms(msg_mb, world_size, fabric) for msg_mb in message_sizes_mb]
    plt.plot(message_sizes_mb, ys, marker="o", label=fabric.name)
plt.xlabel("Message size (MB)")
plt.ylabel("Estimated all-reduce time (ms)")
plt.title("Communication intuition across fabrics")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

def build_torchrun_command(script_name: str = "train.py", nnodes: int = 2, nproc_per_node: int = 8) -> str:
    return f"torchrun --nnodes {nnodes} --nproc_per_node {nproc_per_node} {script_name}"

def build_debug_env() -> Dict[str, str]:
    return {
        "NCCL_DEBUG": "INFO",
        "NCCL_SOCKET_IFNAME": "eth0",
        "TORCH_DISTRIBUTED_DEBUG": "DETAIL",
    }

print("Example launch command:")
print(" ", build_torchrun_command())

print("\nExample debug environment:")
print(json.dumps(build_debug_env(), indent=2))

synthetic_nccl_logs = {
    "interface_mismatch": '''
NCCL INFO NET/Socket : Using [0]eth1:10.0.0.2<0>
NCCL WARN Connect to 10.0.1.9<0> failed : No route to host
''',
    "collective_hang": '''
NCCL INFO rank 3: allReduce start
NCCL INFO rank 7: waiting for allReduce peer
''',
    "cuda_error": '''
NCCL WARN Cuda failure 'out of memory'
NCCL WARN Call to cudaMalloc failed
''',
}

def triage_nccl_log(log_text: str) -> List[str]:
    text = log_text.lower()
    suggestions = []

    if "no route to host" in text or "connect to" in text:
        suggestions.append("Check NCCL_SOCKET_IFNAME, routing, firewalls, and interface selection.")
    if "waiting for allreduce peer" in text or "hang" in text:
        suggestions.append("Check for mismatched collectives, rank crashes, or one rank skipping a synchronization.")
    if "out of memory" in text or "cudamalloc failed" in text:
        suggestions.append("Check per-rank memory, batch size, sequence length, and sharding/checkpointing options.")
    if "warn" in text and not suggestions:
        suggestions.append("Enable more detailed rank-local logging and compare all ranks step-by-step.")

    return suggestions

for name, log_text in synthetic_nccl_logs.items():
    print(f"Scenario: {name}")
    for suggestion in triage_nccl_log(log_text):
        print("  -", suggestion)
    print()



## 5. Distributed checkpointing intuition

Real distributed training often wants:
- sharded save/load
- manifest metadata
- optimizer-state shards
- world-size-flexible restore

PyTorch Distributed Checkpoint (DCP) is the official PyTorch direction, but here we build a local analogy so the concept is executable.


In [ ]:

# ============================================================
# Distributed checkpointing intuition
# ============================================================

def shard_state_dict_by_keys(state_dict: Dict[str, torch.Tensor], num_shards: int) -> List[Dict[str, torch.Tensor]]:
    shards = [dict() for _ in range(num_shards)]
    for idx, key in enumerate(sorted(state_dict.keys())):
        shard_idx = idx % num_shards
        shards[shard_idx][key] = state_dict[key].detach().cpu()
    return shards

def save_sharded_checkpoint(model: nn.Module, checkpoint_dir: Path, num_shards: int = 2) -> Dict[str, object]:
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    state_dict = model.state_dict()
    shards = shard_state_dict_by_keys(state_dict, num_shards)

    manifest = {"num_shards": num_shards, "shards": []}

    for shard_idx, shard in enumerate(shards):
        shard_path = checkpoint_dir / f"model_shard_{shard_idx:02d}.pt"
        torch.save(shard, shard_path)
        manifest["shards"].append(
            {
                "shard_index": shard_idx,
                "path": shard_path.name,
                "keys": sorted(list(shard.keys())),
            }
        )

    manifest_path = checkpoint_dir / "manifest.json"
    with manifest_path.open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    return manifest

def load_sharded_checkpoint(checkpoint_dir: Path) -> Dict[str, torch.Tensor]:
    manifest_path = checkpoint_dir / "manifest.json"
    with manifest_path.open("r", encoding="utf-8") as f:
        manifest = json.load(f)

    merged = {}
    for shard_info in manifest["shards"]:
        shard_path = checkpoint_dir / shard_info["path"]
        shard_state = torch.load(shard_path, map_location="cpu")
        merged.update(shard_state)

    return merged

# Use eval mode to avoid dropout / train-mode noise during equivalence check.
original_checkpoint_model = copy.deepcopy(toy_model).cpu().eval()
checkpoint_dir = Path(tempfile.mkdtemp(prefix="toy_sharded_checkpoint_"))

manifest = save_sharded_checkpoint(original_checkpoint_model, checkpoint_dir, num_shards=2)

print("Checkpoint directory:", checkpoint_dir)
print("Manifest preview:")
print(json.dumps(manifest, indent=2)[:900])

reloaded_model = copy.deepcopy(original_checkpoint_model).cpu().eval()
merged_state = load_sharded_checkpoint(checkpoint_dir)
reloaded_model.load_state_dict(merged_state)

test_input_ids, _ = make_counting_batch(batch_size=4, device=torch.device("cpu"))

with torch.no_grad():
    original_logits = original_checkpoint_model(test_input_ids)
    reloaded_logits = reloaded_model(test_input_ids)

checkpoint_equal = torch.allclose(original_logits, reloaded_logits, atol=1e-6, rtol=1e-5)
print("\nReloaded checkpoint matches original outputs:", checkpoint_equal)



## 6. Step-time analysis, observability, and a tiny profiler pass

A senior habit is:
1. instrument first
2. profile second

Timer-based observability often tells you enough to decide whether you need deeper profiling.


In [ ]:

# ============================================================
# Step-time analysis, observability, and a tiny profiler pass
# ============================================================
# This section intentionally uses a *smaller* model than the main
# toy model so it runs quickly in CPU-only environments.

@dataclass
class StepMetrics:
    data_ms: float
    forward_ms: float
    backward_ms: float
    optimizer_ms: float
    total_ms: float
    tokens_per_s: float
    loss: float

def run_timed_train_step(model: nn.Module, optimizer: torch.optim.Optimizer, loss_fn: nn.Module) -> StepMetrics:
    model.train()

    step_start = time.perf_counter()

    data_start = time.perf_counter()
    input_ids, target_ids = make_counting_batch(batch_size=4, min_digits=4, max_digits=8, device=device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    data_ms = (time.perf_counter() - data_start) * 1000.0

    forward_start = time.perf_counter()
    logits = model(input_ids)
    loss = loss_fn(logits.reshape(-1, VOCAB_SIZE), target_ids.reshape(-1))
    if device.type == "cuda":
        torch.cuda.synchronize()
    forward_ms = (time.perf_counter() - forward_start) * 1000.0

    backward_start = time.perf_counter()
    optimizer.zero_grad()
    loss.backward()
    if device.type == "cuda":
        torch.cuda.synchronize()
    backward_ms = (time.perf_counter() - backward_start) * 1000.0

    optimizer_start = time.perf_counter()
    optimizer.step()
    if device.type == "cuda":
        torch.cuda.synchronize()
    optimizer_ms = (time.perf_counter() - optimizer_start) * 1000.0

    total_ms = (time.perf_counter() - step_start) * 1000.0

    num_tokens = int((target_ids != PAD_ID).sum().item())
    tokens_per_s = num_tokens / (total_ms / 1000.0) if total_ms > 0 else float("nan")

    return StepMetrics(
        data_ms=data_ms,
        forward_ms=forward_ms,
        backward_ms=backward_ms,
        optimizer_ms=optimizer_ms,
        total_ms=total_ms,
        tokens_per_s=tokens_per_s,
        loss=float(loss.item()),
    )

observability_model = TinyCausalLM(
    vocab_size=VOCAB_SIZE,
    d_model=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout=0.0,
    max_len=64,
).to(device)

observability_optimizer = torch.optim.Adam(observability_model.parameters(), lr=2e-3)
observability_loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)

metrics_history = []
for _ in range(2):
    metrics_history.append(run_timed_train_step(observability_model, observability_optimizer, observability_loss_fn))

avg_data_ms = statistics.mean(m.data_ms for m in metrics_history)
avg_forward_ms = statistics.mean(m.forward_ms for m in metrics_history)
avg_backward_ms = statistics.mean(m.backward_ms for m in metrics_history)
avg_optimizer_ms = statistics.mean(m.optimizer_ms for m in metrics_history)
avg_total_ms = statistics.mean(m.total_ms for m in metrics_history)
avg_tokens_per_s = statistics.mean(m.tokens_per_s for m in metrics_history)

print("Average step metrics across 2 steps")
print(f"  data_ms:       {avg_data_ms:.2f}")
print(f"  forward_ms:    {avg_forward_ms:.2f}")
print(f"  backward_ms:   {avg_backward_ms:.2f}")
print(f"  optimizer_ms:  {avg_optimizer_ms:.2f}")
print(f"  total_ms:      {avg_total_ms:.2f}")
print(f"  tokens_per_s:  {avg_tokens_per_s:.2f}")

plt.figure(figsize=(7, 2.8))
left = 0.0
for name, value in zip(
    ["data", "forward", "backward", "optimizer"],
    [avg_data_ms, avg_forward_ms, avg_backward_ms, avg_optimizer_ms],
):
    plt.barh(["avg step"], [value], left=left, label=name)
    left += value
plt.xlabel("Milliseconds")
plt.title("Average step breakdown")
plt.legend()
plt.tight_layout()
plt.show()

# Tiny profiler pass.
from torch.profiler import ProfilerActivity, profile

activities = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    record_shapes=False,
    profile_memory=False,
    with_stack=False,
) as prof:
    _ = run_timed_train_step(observability_model, observability_optimizer, observability_loss_fn)

sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
prof_table = prof.key_averages().table(sort_by=sort_key, row_limit=8)
print(prof_table)



## 7. Failure scenarios and mitigation patterns

Advanced infrastructure interviews often test whether you can move from:
- symptoms
to
- plausible causes
to
- first checks
to
- mitigations

The code below is a small rule-based triage helper to practice that habit.


In [ ]:

# ============================================================
# Failure scenarios and mitigation patterns
# ============================================================

@dataclass
class FailureScenario:
    name: str
    symptoms: List[str]
    likely_causes: List[str]
    first_checks: List[str]
    mitigations: List[str]

FAILURE_PLAYBOOK = [
    FailureScenario(
        name="distributed_hang_on_first_backward",
        symptoms=["all ranks stuck", "hang", "backward starts", "waiting forever"],
        likely_causes=[
            "mismatched collective calls across ranks",
            "one rank crashed earlier",
            "incorrect rendezvous or rank environment",
        ],
        first_checks=[
            "Enable NCCL_DEBUG=INFO and TORCH_DISTRIBUTED_DEBUG=DETAIL",
            "Verify every rank entered the same forward/backward path",
            "Check rank-local logs for an earlier exception",
        ],
        mitigations=[
            "Fix control-flow divergence across ranks",
            "Harden launch and rendezvous validation",
            "Add step boundary logging per rank",
        ],
    ),
    FailureScenario(
        name="gpu_oom_after_longer_context",
        symptoms=["oom", "context length", "memory spike", "longer context"],
        likely_causes=[
            "activation growth from longer sequence length",
            "attention-cost growth",
            "insufficient sharding/checkpointing",
        ],
        first_checks=[
            "Reduce sequence length or batch size",
            "Enable activation checkpointing",
            "Consider stronger sharding / sequence parallelism",
        ],
        mitigations=[
            "Checkpoint more layers",
            "Increase sharding or offload",
            "Re-balance microbatch vs grad-accumulation",
        ],
    ),
    FailureScenario(
        name="input_pipeline_bottleneck",
        symptoms=["gpu idle", "data time huge", "slow dataloader"],
        likely_causes=[
            "slow preprocessing",
            "storage bottleneck",
            "CPU worker inefficiency",
        ],
        first_checks=[
            "Measure data time explicitly",
            "Profile dataloader and preprocessing path",
            "Compare tokens/s with synthetic in-memory data",
        ],
        mitigations=[
            "Optimize preprocessing",
            "Cache or precompute transforms",
            "Improve host-side pipeline parallelism",
        ],
    ),
]

def diagnose_training_issue(observed_text: str) -> List[Dict[str, object]]:
    text = observed_text.lower()
    matches = []

    for scenario in FAILURE_PLAYBOOK:
        score = 0
        for symptom in scenario.symptoms:
            if symptom in text:
                score += 1

        if score > 0:
            matches.append(
                {
                    "name": scenario.name,
                    "score": score,
                    "likely_causes": scenario.likely_causes,
                    "first_checks": scenario.first_checks,
                    "mitigations": scenario.mitigations,
                }
            )

    matches.sort(key=lambda row: row["score"], reverse=True)
    return matches

triage_examples = [
    "All ranks hang after backward starts.",
    "OOM appears after longer context length.",
    "GPU idle and data time huge.",
]

for example in triage_examples:
    print("Observed problem:", example)
    diagnoses = diagnose_training_issue(example)
    for row in diagnoses[:2]:
        print("  scenario:", row["name"])
        print("  first checks:")
        for item in row["first_checks"]:
            print("   -", item)
    print()



## 8. Interview-style exercises

### Exercise 1 — ZeRO tradeoff

If a model fits under ZeRO-2 but not under ZeRO-1, what does that tell you?

**Hint:**  
Ask which memory category changed between the two stages.



### Exercise 1 — Answer

It suggests that **gradient memory** was an important bottleneck.

Why:
- ZeRO-1 shards optimizer state only
- ZeRO-2 shards optimizer state **and gradients**



### Exercise 2 — Offload tradeoff

Why can optimizer offload help a memory-bound run but still reduce throughput?

**Hint:**  
Think about where optimizer state lives and how often it is touched.



### Exercise 2 — Answer

Offloading optimizer state to CPU or slower storage reduces GPU memory usage.

But that state is still needed during optimization, so offload introduces:
- extra data movement
- extra latency
- lower effective overlap in some cases



### Exercise 3 — Tensor parallelism reasoning

Why does row-parallel linear need a reduction across shards, while column-parallel linear can often concatenate outputs?

**Hint:**  
Think about which dimension is partitioned.



### Exercise 3 — Answer

In column-parallel linear, each shard produces a different slice of output features, so concatenation is natural.

In row-parallel linear, each shard computes a partial contribution to the same output features, so those partial outputs must be summed.



### Exercise 4 — Sequence parallelism reasoning

Why is LayerNorm a natural example for sequence parallelism?

**Hint:**  
Ask whether one token's normalization depends on another token.



### Exercise 4 — Answer

LayerNorm is token-local over the hidden dimension.

That means one token's normalization does not need another token's data, so sharding along the sequence dimension preserves correctness for this kind of operation.



### Exercise 5 — Expert parallelism reasoning

Why can MoE training have a load-balance problem even if routing is "correct"?

**Hint:**  
Correct routing and balanced routing are not the same thing.



### Exercise 5 — Answer

A router can be correct in the sense that tokens follow the gating decision, but still be imbalanced if too many tokens choose the same expert.

That creates:
- overloaded experts
- idle experts
- capacity overflow / dropped tokens
- lower hardware efficiency



### Exercise 6 — NCCL debugging

A multi-node job hangs at the first all-reduce. What are your first checks?

**Hint:**  
Think about collectives, rank health, and interface selection.



### Exercise 6 — Answer

Good first checks:
- enable `NCCL_DEBUG=INFO` and `TORCH_DISTRIBUTED_DEBUG=DETAIL`
- verify every rank entered the same collective path
- inspect logs for an earlier rank-local crash
- verify interface selection such as `NCCL_SOCKET_IFNAME`
- confirm rendezvous, world size, and rank values



### Exercise 7 — Distributed checkpointing

Why are sharded checkpoints often a better fit than one giant checkpoint file in distributed training?

**Hint:**  
Think about existing sharded state and restart workflows.



### Exercise 7 — Answer

Sharded checkpoints fit distributed training better because:
- model and optimizer state may already be sharded
- per-rank parallel I/O scales better
- restart logic can reshard across different world sizes
- one giant file can become a bottleneck or a failure hotspot



### Exercise 8 — Small coding challenge

Modify the expert-router `capacity_factor` and rerun the routing section.

What happens to:
- dropped tokens
- expert loads
- accepted tokens per expert

**Hint:**  
Try a value smaller than `1.0` first, then larger than `1.0`.



### Exercise 8 — Answer

Lower capacity factors usually:
- increase dropped tokens
- make capacity bottlenecks more obvious

Higher capacity factors usually:
- reduce dropped tokens
- allow experts to accept more tokens

But a higher capacity factor does not automatically fix poor routing balance.



## 9. Final takeaways

For advanced L5/L6-style training-infra discussions, try to be able to explain these clearly:

1. what each ZeRO stage removes from GPU replication  
2. why offload is a memory/throughput tradeoff  
3. how tensor parallel, sequence parallel, and expert parallel solve different problems  
4. why multi-node communication is often a fabric + collective + debugging problem  
5. why distributed checkpoints need manifests and shard-aware thinking  
6. why step-time observability usually comes before heavy profiling  
7. how to move from symptoms to structured triage and mitigation  



## 10. Suggested next steps

After this notebook, strong follow-ups are:
- run a real `torchrun` DDP or FSDP job on multiple GPUs
- try DeepSpeed or a Megatron-style training stack
- practice reading NCCL logs from a real distributed failure
- experiment with real distributed checkpointing and resume
- pair timer-based observability with a deeper profiler workflow

This notebook is meant to make those next steps much easier.
